# Deep Reinforcement Learning — DQN

> Based on Stanford CS230 — LN9, Mnih et al. (2015) — *Human-level control through deep reinforcement learning*

Reinforcement learning trains an agent to maximise cumulative reward through interaction with an environment. **Deep Q-Networks (DQN)** combine Q-learning with deep neural networks and two key stabilisation tricks: **experience replay** and a **target network**.

## Learning Objectives

1. Define the Markov Decision Process (MDP) framework
2. Write the Bellman equation for Q-values
3. Explain experience replay and the target network
4. Implement DQN and train on a toy environment (CartPole-style)

## Markov Decision Process

$(\mathcal{S}, \mathcal{A}, \mathcal{R}, \mathcal{P}, \gamma)$

- $s_t \in \mathcal{S}$ — state at time $t$
- $a_t \in \mathcal{A}$ — action
- $r_t = \mathcal{R}(s_t, a_t)$ — reward
- $\mathcal{P}(s_{t+1} \mid s_t, a_t)$ — transition dynamics
- $\gamma \in [0,1)$ — discount factor

**Goal**: find policy $\pi(a \mid s)$ maximising $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$

## Q-Learning & Bellman Equation

$$Q^*(s,a) = \mathbb{E}\!\left[r + \gamma \max_{a'} Q^*(s', a') \;\middle|\; s,a\right]$$

**DQN loss** (TD error):

$$\mathcal{L}(\theta) = \mathbb{E}_{\langle s,a,r,s'\rangle \sim \mathcal{D}}\!\left[\bigl(r + \gamma \max_{a'} Q(s', a'; \theta^-) - Q(s,a;\theta)\bigr)^2\right]$$

- $\theta$ — online network parameters (updated every step)
- $\theta^-$ — target network parameters (frozen, copied from $\theta$ every $C$ steps)
- $\mathcal{D}$ — replay buffer

## DQN Architecture

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 170" width="680" height="170" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="da" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Environment -->
  <rect x="10" y="60" width="100" height="50" rx="5" fill="#56b6c2" fill-opacity="0.15" stroke="#56b6c2" stroke-width="1.5"/>
  <text x="60" y="82" text-anchor="middle" fill="#2a7080" font-size="10" font-weight="bold">Environment</text>
  <text x="60" y="97" text-anchor="middle" fill="#2a7080" font-size="9">s, r, done</text>
  <!-- Replay Buffer -->
  <rect x="165" y="100" width="120" height="40" rx="4" fill="#f4b942" fill-opacity="0.18" stroke="#f4b942" stroke-width="1.5"/>
  <text x="225" y="122" text-anchor="middle" fill="#a07010" font-size="10" font-weight="bold">Replay Buffer D</text>
  <text x="225" y="134" text-anchor="middle" fill="#a07010" font-size="9">(s,a,r,s',done)</text>
  <!-- Online Q-network -->
  <rect x="340" y="40" width="140" height="55" rx="5" fill="#5b9bd5" fill-opacity="0.18" stroke="#5b9bd5" stroke-width="1.6"/>
  <text x="410" y="63" text-anchor="middle" fill="#3a7bbf" font-size="10" font-weight="bold">Online Q-net θ</text>
  <text x="410" y="78" text-anchor="middle" fill="#3a7bbf" font-size="9">Q(s,a;θ) for all a</text>
  <text x="410" y="89" text-anchor="middle" fill="#3a7bbf" font-size="9">→ ε-greedy action</text>
  <!-- Target Q-network -->
  <rect x="340" y="110" width="140" height="45" rx="5" fill="#c678dd" fill-opacity="0.12" stroke="#c678dd" stroke-width="1.5"/>
  <text x="410" y="130" text-anchor="middle" fill="#8a3ab8" font-size="10" font-weight="bold">Target Q-net θ⁻</text>
  <text x="410" y="144" text-anchor="middle" fill="#8a3ab8" font-size="9">θ⁻ ← θ every C steps</text>
  <!-- Loss / TD error -->
  <rect x="530" y="65" width="130" height="55" rx="5" fill="#e05c5c" fill-opacity="0.12" stroke="#e05c5c" stroke-width="1.5"/>
  <text x="595" y="88" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">TD Error</text>
  <text x="595" y="102" text-anchor="middle" fill="#b03a3a" font-size="9">(y - Q(s,a;θ))²</text>
  <text x="595" y="114" text-anchor="middle" fill="#b03a3a" font-size="9">→ update θ</text>
  <!-- Arrows -->
  <line x1="110"  y1="85"  x2="165" y2="122" stroke="#aaa" stroke-width="1.2" marker-end="url(#da)"/>
  <line x1="110"  y1="75"  x2="340" y2="65"  stroke="#aaa" stroke-width="1.2" marker-end="url(#da)"/>
  <line x1="225"  y1="100" x2="340" y2="70"  stroke="#aaa" stroke-width="1.2" marker-end="url(#da)"/>
  <line x1="225"  y1="100" x2="340" y2="135" stroke="#aaa" stroke-width="1.2" marker-end="url(#da)"/>
  <line x1="480"  y1="67"  x2="530" y2="87"  stroke="#aaa" stroke-width="1.2" marker-end="url(#da)"/>
  <line x1="480"  y1="133" x2="530" y2="110" stroke="#aaa" stroke-width="1.2" marker-end="url(#da)"/>
  <!-- Action feedback -->
  <line x1="410"  y1="40"  x2="410" y2="20"  stroke="#5b9bd5" stroke-width="1.4"/>
  <line x1="410"  y1="20"  x2="60"  y2="20"  stroke="#5b9bd5" stroke-width="1.4"/>
  <line x1="60"   y1="20"  x2="60"  y2="60"  stroke="#5b9bd5" stroke-width="1.4" marker-end="url(#da)"/>
  <text x="230" y="16" text-anchor="middle" fill="#3a7bbf" font-size="9">action a (ε-greedy)</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)

# ---- Toy 1-D environment: reach goal ----
# State: position ∈ {0..9}, Action: 0=left, 1=right
# Reward: +1 at position 9, -0.01 step cost, episode ends at 9 or after 20 steps

class GridEnv:
    def __init__(self, n=10):
        self.n    = n
        self.goal = n - 1

    def reset(self):
        self.pos   = np.random.randint(0, self.n - 1)
        self.steps = 0
        return self._obs()

    def _obs(self):
        s = np.zeros(self.n)
        s[self.pos] = 1.0
        return s

    def step(self, action):
        self.steps += 1
        if action == 0:
            self.pos = max(0, self.pos - 1)
        else:
            self.pos = min(self.n - 1, self.pos + 1)
        done   = (self.pos == self.goal) or (self.steps >= 20)
        reward = 1.0 if self.pos == self.goal else -0.01
        return self._obs(), reward, done

# ---- Q-Network ----
class QNet:
    def __init__(self, n_s, n_a, n_h=32, seed=0):
        np.random.seed(seed)
        s = 0.1
        self.W1 = np.random.randn(n_h, n_s) * s; self.b1 = np.zeros(n_h)
        self.W2 = np.random.randn(n_a, n_h) * s; self.b2 = np.zeros(n_a)

    def predict(self, s):
        z1 = self.W1 @ s + self.b1
        a1 = relu(z1)
        return self.W2 @ a1 + self.b2, a1, z1

    def copy_weights_from(self, other):
        self.W1 = other.W1.copy(); self.b1 = other.b1.copy()
        self.W2 = other.W2.copy(); self.b2 = other.b2.copy()

    def update(self, s, action, target, lr):
        q_vals, a1, z1 = self.predict(s)
        # Compute gradient of (target - q(s,a))^2 w.r.t. weights
        td = q_vals[action] - target
        dq = np.zeros_like(q_vals); dq[action] = td
        d_W2 = dq[:, None] * a1[None, :]
        d_b2 = dq
        d_a1 = self.W2.T @ dq
        d_z1 = d_a1 * relu_d(z1)
        d_W1 = d_z1[:, None] * s[None, :]
        d_b1 = d_z1
        self.W1 -= lr * d_W1
        self.b1 -= lr * d_b1
        self.W2 -= lr * d_W2
        self.b2 -= lr * d_b2
        return 0.5 * td**2

# ---- DQN training loop ----
def train_dqn(n_episodes=300, gamma=0.95, lr=0.01,
              eps_start=1.0, eps_end=0.05, eps_decay=0.99,
              buf_size=1000, batch=32, target_update=10, seed=1):
    np.random.seed(seed)
    env    = GridEnv()
    n_s, n_a = env.n, 2
    online = QNet(n_s, n_a, seed=0)
    target = QNet(n_s, n_a, seed=0); target.copy_weights_from(online)
    buffer = deque(maxlen=buf_size)
    eps    = eps_start
    ep_rewards, losses = [], []

    for ep in range(n_episodes):
        s = env.reset()
        done = False; total_r = 0
        while not done:
            # ε-greedy action
            if np.random.rand() < eps:
                a = np.random.randint(n_a)
            else:
                q, _, _ = online.predict(s)
                a = q.argmax()
            s2, r, done = env.step(a)
            buffer.append((s.copy(), a, r, s2.copy(), done))
            total_r += r
            s = s2

            # Sample minibatch and update
            if len(buffer) >= batch:
                idx  = np.random.choice(len(buffer), batch, replace=False)
                loss = 0
                for i in idx:
                    bs, ba, br, bs2, bd = buffer[i]
                    q2, _, _ = target.predict(bs2)
                    td_target = br + (0 if bd else gamma * q2.max())
                    loss += online.update(bs, ba, td_target, lr)
                losses.append(loss / batch)

        ep_rewards.append(total_r)
        eps = max(eps_end, eps * eps_decay)
        if ep % target_update == 0:
            target.copy_weights_from(online)

    return ep_rewards, losses, online

ep_rewards, losses, final_net = train_dqn()

# ---- Plots ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('DQN on 1-D Grid Environment', fontsize=13, fontweight='bold')

# Episode rewards
window = 20
smooth = np.convolve(ep_rewards, np.ones(window)/window, 'valid')
axes[0].plot(ep_rewards, color=P[0], alpha=0.3, lw=1, label='Raw reward')
axes[0].plot(np.arange(window-1, len(ep_rewards)), smooth, color=P[0], lw=2, label=f'{window}-ep avg')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Total reward')
axes[0].set_title('Episode Rewards'); axes[0].legend(); axes[0].grid(True)

# TD losses
if losses:
    w2 = 30
    sl = np.convolve(losses, np.ones(w2)/w2, 'valid')
    axes[1].plot(losses, color=P[2], alpha=0.3, lw=1)
    axes[1].plot(sl, color=P[2], lw=2)
    axes[1].set_xlabel('Training step'); axes[1].set_ylabel('TD Loss')
    axes[1].set_title('TD Error (MSE)'); axes[1].grid(True)

# Q-value heatmap for final policy
states = np.eye(GridEnv().n)
q_table = np.array([final_net.predict(s)[0] for s in states])
im = axes[2].imshow(q_table.T, cmap='RdYlGn', aspect='auto')
axes[2].set_xticks(range(10)); axes[2].set_xticklabels([f's{i}' for i in range(10)])
axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(['← left', '→ right'])
axes[2].set_xlabel('State'); axes[2].set_title('Learned Q-values\n(green=high, red=low)')
plt.colorbar(im, ax=axes[2], label='Q(s,a)')

# Overlay greedy policy arrows
for s in range(10):
    best_a = q_table[s].argmax()
    arrow  = '→' if best_a == 1 else '←'
    color  = P[3] if best_a == 1 else P[1]
    axes[2].text(s, 2.15, arrow, ha='center', color=color, fontsize=14, fontweight='bold',
                 transform=axes[2].get_xaxis_transform())

plt.tight_layout()
plt.show()

# ---- ε decay schedule ----
eps_vals = [max(0.05, 1.0 * 0.99**ep) for ep in range(300)]
fig, ax  = plt.subplots(figsize=(9, 3))
ax.plot(eps_vals, color=P[5], lw=2)
ax.set_xlabel('Episode'); ax.set_ylabel('ε (exploration rate)')
ax.set_title('ε-Greedy Exploration Decay (eps_decay=0.99)')
ax.fill_between(range(300), eps_vals, alpha=0.15, color=P[5])
ax.grid(True)
plt.tight_layout()
plt.show()
